# ERA5 - Processing from 2 types : 
1 - ERA5 Monthly average
2 - ERA5 Monthly extrema (max and min) - built from ERA5 hourly data

# 0 - Initialization

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr


# ============================================================
# CONFIGURATION
# ============================================================

RAW_DIR = Path("/home/mbaldacchino/data/era5_hourly")
PROCESSED_DIR = Path("/home/mbaldacchino/data/era5_monthly_extrema")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CLIMATE_FILE = Path("/home/mbaldacchino/data/era5_monthly_1970_2026")
CLIMATE_FILE_PROCESSED = Path("/home/mbaldacchino/data/era5_monthly_1970_2026.nc")
EXTREMA_FILE_PROCESSED = Path("/home/mbaldacchino/data/era5_monthly_extrema_1970_2026.nc")

ERA5_MONTHLY_FILE = Path("/home/mbaldacchino/data/era5_monthly_averaged_1970_2026.grib")

FINAL_FILE = Path("/home/mbaldacchino/data/era5_monthly_complete_1970_2026.nc")


# 1 - Functions

In [ ]:
def open_era5_extrema_grib(path: str | Path) -> xr.Dataset:
    """
    Open an ERA5 mx2t/mn2t GRIB and flatten the
    (time, step) forecast representation into a single
    hourly valid_time dimension.
    """

    path = Path(path)

    ds = xr.open_dataset(
        path,
        engine="cfgrib",
        backend_kwargs={"indexpath": ""},
    )

    required = {"mx2t", "mn2t"}

    missing = required - set(ds.data_vars)

    if missing:
        raise ValueError(
            f"{path.name}: missing variables {missing}. Available variables: {list(ds.data_vars)}"
        )

    # Keep only variables of interest
    ds = ds[["mx2t", "mn2t"]]

    # --------------------------------------------------------
    # Flatten:
    #
    # time × step
    #     ->
    # sample
    #
    # valid_time automatically becomes valid_time(sample)
    # during stack().
    # --------------------------------------------------------

    ds = ds.stack(sample=("time", "step"))

    # IMPORTANT:
    # valid_time already exists at this stage.
    #
    # Therefore use it as the dimension instead of trying
    # to rename "sample" -> "valid_time".
    ds = ds.swap_dims({"sample": "valid_time"})

    # Remove the now-unnecessary forecast coordinates
    ds = ds.drop_vars(
        ["sample", "time", "step"],
        errors="ignore",
    )

    # Put dimensions into a convenient order
    ds = ds.transpose(
        "valid_time",
        "latitude",
        "longitude",
    )

    ds = ds.sortby("valid_time")

    # --------------------------------------------------------
    # Sanity checks
    # --------------------------------------------------------

    times = pd.DatetimeIndex(ds.valid_time.values)

    if times.has_duplicates:
        duplicates = times[times.duplicated()].unique()

        raise ValueError(f"{path.name}: duplicate valid times found: {duplicates[:10]}")

    time_diffs = np.diff(times.values)

    if len(time_diffs) > 0:
        if not np.all(time_diffs == np.timedelta64(1, "h")):
            raise ValueError(f"{path.name}: valid_time is not continuously hourly.")

    return ds


def hourly_to_monthly_extrema(
    ds: xr.Dataset,
) -> xr.Dataset:
    """
    Convert hourly ERA5 mx2t/mn2t interval extrema into:

    - monthly mean of daily maximum 2m temperature
    - monthly mean of daily minimum 2m temperature

    Only complete days (24 hourly intervals) and complete
    calendar months are retained.
    """

    # --------------------------------------------------------
    # Assign each interval to its starting hour
    #
    # valid_time = 00:00 represents 23:00 -> 00:00,
    # therefore associate it with 23:00 of the previous day.
    # --------------------------------------------------------

    ds = ds.assign_coords(valid_time=(ds.valid_time - np.timedelta64(1, "h")))

    # --------------------------------------------------------
    # Count number of hourly intervals per calendar day
    # --------------------------------------------------------

    hourly_counter = xr.DataArray(
        np.ones(
            ds.sizes["valid_time"],
            dtype=np.uint8,
        ),
        coords={"valid_time": ds.valid_time},
        dims="valid_time",
    )

    daily_count = hourly_counter.resample(valid_time="1D").sum()

    complete_day = daily_count == 24

    # --------------------------------------------------------
    # Hourly -> daily extrema
    # --------------------------------------------------------

    daily_max = ds["mx2t"].resample(valid_time="1D").max().where(complete_day)

    daily_min = ds["mn2t"].resample(valid_time="1D").min().where(complete_day)

    # --------------------------------------------------------
    # Daily -> monthly
    # --------------------------------------------------------

    monthly_max = daily_max.resample(valid_time="MS").mean()

    monthly_min = daily_min.resample(valid_time="MS").mean()

    monthly = xr.Dataset(
        {
            "t2m_max": monthly_max,
            "t2m_min": monthly_min,
        }
    )

    # --------------------------------------------------------
    # Only retain COMPLETE months
    #
    # This is especially useful for:
    # - first/last month of each GRIB
    # - current month in 2026
    # --------------------------------------------------------

    complete_days_per_month = complete_day.astype(np.uint8).resample(valid_time="MS").sum()

    month_dates = pd.DatetimeIndex(complete_days_per_month.valid_time.values)

    expected_days = xr.DataArray(
        month_dates.days_in_month,
        coords={"valid_time": complete_days_per_month.valid_time},
        dims="valid_time",
    )

    complete_month = (complete_days_per_month == expected_days).values

    monthly = monthly.isel(valid_time=complete_month)

    # Use conventional monthly dimension name
    monthly = monthly.rename({"valid_time": "time"})

    monthly["t2m_max"].attrs = {
        "long_name": "Monthly mean of daily maximum 2m temperature",
        "units": ds["mx2t"].attrs.get("units", "K"),
    }

    monthly["t2m_min"].attrs = {
        "long_name": "Monthly mean of daily minimum 2m temperature",
        "units": ds["mn2t"].attrs.get("units", "K"),
    }

    return monthly


def process_era5_extrema_file(
    input_path: str | Path,
    output_path: str | Path,
) -> None:
    """
    Process one ERA5 hourly extrema GRIB and export
    monthly mean daily Tmax/Tmin as NetCDF.
    """

    input_path = Path(input_path)
    output_path = Path(output_path)

    print(f"Processing: {input_path.name}")

    ds = open_era5_extrema_grib(input_path)

    monthly = hourly_to_monthly_extrema(ds)

    encoding = {
        var: {
            "dtype": "float32",
            "zlib": True,
            "complevel": 4,
        }
        for var in monthly.data_vars
    }

    monthly.to_netcdf(
        output_path,
        encoding=encoding,
    )

    print(f"  {monthly.time.values[0]} -> {monthly.time.values[-1]}")

    print(f"  {monthly.sizes['time']} months")

    print(f"  saved: {output_path}")

    ds.close()
    monthly.close()

def combine_monthly_extrema(
    processed_files: list[Path],
    output_path: str | Path,
) -> xr.Dataset:
    """
    Concatenate all processed monthly extrema files into
    one continuous monthly ERA5 dataset.
    """

    output_path = Path(output_path)

    datasets = [xr.open_dataset(path) for path in processed_files]

    ds = xr.concat(
        datasets,
        dim="time",
        data_vars="minimal",
        coords="minimal",
        compat="override",
    )

    ds = ds.sortby("time")

    # --------------------------------------------------------
    # Check duplicate months
    # --------------------------------------------------------

    dates = pd.DatetimeIndex(ds.time.values)

    if dates.has_duplicates:
        duplicates = dates[dates.duplicated()].unique()

        raise ValueError(f"Duplicate months found between processed files: {duplicates}")

    # --------------------------------------------------------
    # Check continuity
    # --------------------------------------------------------

    periods = dates.to_period("M")

    expected = pd.period_range(
        periods.min(),
        periods.max(),
        freq="M",
    )

    missing = expected.difference(periods)

    if len(missing) > 0:
        raise ValueError(f"Missing months in the combined dataset:\n{list(missing)}")

    encoding = {
        var: {
            "dtype": "float32",
            "zlib": True,
            "complevel": 4,
        }
        for var in ds.data_vars
    }

    ds.to_netcdf(
        output_path,
        encoding=encoding,
    )

    print(f"Combined dataset: {dates.min():%Y-%m} -> {dates.max():%Y-%m}")

    print(f"{len(dates)} months")

    print(f"Saved: {output_path}")

    for dataset in datasets:
        dataset.close()

    return xr.open_dataset(output_path)

def normalize_monthly_time(
    ds: xr.Dataset,
) -> xr.Dataset:
    """
    Normalize a monthly time coordinate to YYYY-MM-01 00:00.
    """

    dates = pd.DatetimeIndex(ds.time.values)

    month_start = dates.to_period("M").to_timestamp()

    return ds.assign_coords(time=month_start.values)


def check_same_grid(
    ds1: xr.Dataset,
    ds2: xr.Dataset,
) -> None:

    for coord in [
        "latitude",
        "longitude",
    ]:
        if coord not in ds1.coords:
            raise ValueError(f"{coord} missing from dataset 1")

        if coord not in ds2.coords:
            raise ValueError(f"{coord} missing from extrema dataset")

        if ds1.sizes[coord] != ds2.sizes[coord]:
            raise ValueError(f"Different {coord} sizes: {ds1.sizes[coord]} vs {ds2.sizes[coord]}")

        if not np.allclose(
            ds1[coord].values,
            ds2[coord].values,
        ):
            raise ValueError(f"{coord} coordinates differ.")


# 2 - Use

In [ ]:
hourly_files = sorted(list(RAW_DIR.glob("*.grib")) + list(RAW_DIR.glob("*.grib2")))

if not hourly_files:
    raise FileNotFoundError(f"No GRIB files found in {RAW_DIR}")


processed_files = []

for path in hourly_files:
    output_path = PROCESSED_DIR / f"{path.stem}_monthly.nc"

    # process_era5_extrema_file(
    #     input_path=path,
    #     output_path=output_path,
    # )

    processed_files.append(output_path)


In [ ]:
era5_extrema = combine_monthly_extrema(
    processed_files=processed_files,
    output_path=EXTREMA_FILE_PROCESSED,
)

print(era5_extrema)


In [ ]:
# PROCESS THE ERA5 MONTHLY TO COMBINE

processed = list(CLIMATE_FILE.glob("*.nc"))

ds = [xr.open_dataset(p).drop_vars(["expver"]) for p in processed]
ds[1] = ds[1].assign_coords(valid_time=(ds[1].valid_time - np.timedelta64(6, "h")))
ds = ds[0].merge(ds[1])
ds.to_netcdf(
    CLIMATE_FILE_PROCESSED,
    encoding={var: {"dtype": "float32", "zlib": True, "complevel": 4} for var in ds.data_vars},
)


In [ ]:
era5_monthly = xr.open_dataset(CLIMATE_FILE_PROCESSED).rename({"valid_time": "time"})

era5_monthly = normalize_monthly_time(era5_monthly)

era5_extrema = normalize_monthly_time(era5_extrema)

In [ ]:
check_same_grid(
    era5_monthly,
    era5_extrema,
)


In [ ]:
common_time = np.intersect1d(
    era5_monthly.time.values,
    era5_extrema.time.values,
)

if len(common_time) == 0:
    raise ValueError("The two datasets have no common monthly dates.")

era5_monthly = era5_monthly.sel(time=common_time)

era5_extrema = era5_extrema.sel(time=common_time)

era5_final = xr.merge(
    [
        era5_monthly,
        era5_extrema,
    ],
    join="exact",
    compat="no_conflicts",
)

era5_final = era5_final.sortby("time")

era5_final.to_netcdf(
    FINAL_FILE,
    encoding={
        var: {"dtype": "float32", "zlib": True, "complevel": 4} for var in era5_final.data_vars
    },
)
